In [289]:
import pandas as pd
import numpy as np

In [290]:
df = pd.read_csv("C://Users/user/Desktop/pet/pet-projects/ml/churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [291]:
df = df.dropna(inplace=True)

cols = df.select_dtypes(include='object').columns

mask = df[cols].apply(lambda col: col.astype(str).str.strip().eq('')).any(axis=1)

df = df[~mask]

AttributeError: 'NoneType' object has no attribute 'select_dtypes'

In [ ]:
X = df.drop('Churn', axis=1).copy()
y = (df['Churn']).copy()

In [ ]:
X = X.drop('customerID', axis=1)

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
if 'Churn' in categorical_cols:
    categorical_cols.remove('Churn')


X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [ ]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,0.000000,0,1,0,0,2,0,0,0,0,0,1,2,0.115423,0.383520
1,1,0,0,0,0.464789,1,0,0,2,0,2,0,0,0,1,0,3,0.385075,0.224384
2,1,0,0,0,0.014085,1,0,0,2,2,0,0,0,0,0,1,3,0.354229,0.023893
3,1,0,0,0,0.619718,0,1,0,2,0,2,2,0,0,1,0,0,0.239303,0.214275
4,0,0,0,0,0.014085,1,0,1,0,0,0,0,0,0,0,1,2,0.521891,0.141522


In [ ]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(y)



In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

for col in X.columns:
    if X[col].max() > 3:
        st = MinMaxScaler()
        X[col] = st.fit_transform(X[[col]])

In [ ]:
X.shape

(7032, 19)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =train_test_split(
    X,y,
    random_state=42,
    test_size=0.2,
    shuffle=True,
    stratify=y
)

In [ ]:
X_train.shape

(5625, 19)

In [ ]:
c = 0
for i in y_train:
    if i == 1:
        c += 1

c

1495

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, recall_score, precision_score

dummy = DummyClassifier()

dummy.fit(X_train, y_train)
p = dummy.predict(X_test)

f1_score(y_test, p)

0.0

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

pred = (np.array(pred) >= 0.4).astype(int)

print('logreg')
f1_score(y_test, pred)

logreg


0.6078028747433265

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

model1 = LogisticRegression(
    class_weight='balanced',
    random_state=42
)
model1.fit(X_smote, y_smote)
pred1 = model1.predict_proba(X_test)[:, 1]

pred1 = (np.array(pred1) >= 0.4).astype(int)

print('logreg_smote')
f1_score(y_test, pred1)

logreg_smote


0.5804597701149425

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=8,
    min_samples_split=9,
    n_estimators=100,
    )


rf.fit(X_train, y_train)

pred2 = rf.predict_proba(X_test)[:, 1]

pred2 = (np.array(pred2) >= 0.5).astype(int)

print('fr')
f1_score(y_test, pred2)

fr


0.6220302375809935

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=6,
    n_estimators=1000,
    min_samples_split=5,
    )

rf.fit(X_smote, y_smote)

pred3 = rf.predict(X_test)

print('fr_smote')
f1_score(y_test, pred3)

fr_smote


0.6033402922755741

In [ ]:
from catboost import CatBoostClassifier


model3 = CatBoostClassifier(
    random_seed=42,
    learning_rate=0.01,
    loss_function='Logloss',
    eval_metric='F1',
    iterations=2000,
    depth=4,
    l2_leaf_reg=100
    )

model3.fit(X_train, y_train)
pred4 = model3.predict_proba(X_test)[:, 1]


pred4 = (np.array(pred4) >= 0.4).astype(int)
f1_score(y_test, pred4)

0:	learn: 0.4216471	total: 2.96ms	remaining: 5.91s
1:	learn: 0.4240150	total: 6.28ms	remaining: 6.27s
2:	learn: 0.5226337	total: 9.19ms	remaining: 6.12s
3:	learn: 0.5476190	total: 11.8ms	remaining: 5.91s
4:	learn: 0.5495706	total: 15.7ms	remaining: 6.25s
5:	learn: 0.5692367	total: 18.6ms	remaining: 6.2s
6:	learn: 0.5381238	total: 21.3ms	remaining: 6.07s
7:	learn: 0.5587426	total: 24.3ms	remaining: 6.06s
8:	learn: 0.5631825	total: 27.6ms	remaining: 6.1s
9:	learn: 0.5596512	total: 31ms	remaining: 6.16s
10:	learn: 0.5628220	total: 33.6ms	remaining: 6.07s
11:	learn: 0.5737578	total: 36.4ms	remaining: 6.03s
12:	learn: 0.5756286	total: 39.6ms	remaining: 6.05s
13:	learn: 0.5753000	total: 42.5ms	remaining: 6.02s
14:	learn: 0.5802469	total: 45.6ms	remaining: 6.03s
15:	learn: 0.5739737	total: 48.4ms	remaining: 6s
16:	learn: 0.5796767	total: 51ms	remaining: 5.95s
17:	learn: 0.5790285	total: 53.8ms	remaining: 5.92s
18:	learn: 0.5797546	total: 56.5ms	remaining: 5.89s
19:	learn: 0.5760122	total: 59.

0.6180904522613065

In [ ]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,0.000000,0,1,0,0,2,0,0,0,0,0,1,2,0.115423,0.383520
1,1,0,0,0,0.464789,1,0,0,2,0,2,0,0,0,1,0,3,0.385075,0.224384
2,1,0,0,0,0.014085,1,0,0,2,2,0,0,0,0,0,1,3,0.354229,0.023893
3,1,0,0,0,0.619718,0,1,0,2,0,2,2,0,0,1,0,0,0.239303,0.214275
4,0,0,0,0,0.014085,1,0,1,0,0,0,0,0,0,0,1,2,0.521891,0.141522


In [ ]:
cat_fet = X.columns

fet_idx = np.arange(len(cat_fet))
fet_idx = np.delete( fet_idx, [4, 17, 18])

In [ ]:
fet_idx

array([ 0,  1,  2,  3,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16])

In [ ]:
from imblearn.over_sampling import SMOTENC

nc = SMOTENC(categorical_features = fet_idx.tolist(), random_state= 42)

X_nc, y_nc = nc.fit_resample(X_train, y_train)

In [ ]:
model_nc1 = LogisticRegression(
    random_state=42,
    class_weight='balanced',
)

model_nc1.fit(X_nc, y_nc)
pred_nc1 = model_nc1.predict_proba(X_test)[:, 1]
pred_nc1 = (np.array(pred_nc1) >= 0.5).astype(int)

f1_score(y_test, pred_nc1)

0.6080508474576272

In [ ]:
model_nc = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=8,
    min_samples_split=9,
    n_estimators=100,
)

model_nc.fit(X_nc, y_nc)
pred_nc = model_nc.predict_proba(X_test)[:, 1]
pred_nc = (np.array(pred_nc) >= 0.5).astype(int)

f1_score(y_test, pred_nc)




0.610738255033557